<a href="https://colab.research.google.com/github/leovarconrenno/Estacao-de-Reabastecimento-de-Hidrogenio---SCADA-Core/blob/main/etapa-02-grafos/18%20-%20Grafos%20de%20Logistica%20de%20Insumos%20e%20Produtos%20Acabados.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 18 - Notebook: Grafos de Logística de Insumos e Produtos Acabados (SCADA H₂)

Este notebook complementa a malha interna de processo das aulas anteriores com dois modelos de redes logísticas da Estação de Reabastecimento de Hidrogênio: um dígrafo acíclico (DAG) para o **fluxo de materiais e utilidades** ($G_M$) e outro para o **anel viário de circulação interna de veículos** ($G_V$).

![Grafo da Estação de Hidrogênio](grafo_tubulacoes.png)

In [1]:
import sys
sys.stdout.reconfigure(encoding='utf-8')

from dataclasses import dataclass
from heapq import heappop, heappush
from math import inf
from typing import Dict, List, Tuple


@dataclass(frozen=True)
class Rota:
    destino: str
    peso: float
    descricao: str


class GrafoLogistico:
    def __init__(self) -> None:
        self._adjacencias: Dict[str, List[Rota]] = {}

    def adicionar_rota(self, origem: str, destino: str, peso: float, descricao: str) -> None:
        if peso < 0:
            raise ValueError('Dijkstra exige pesos nao negativos.')
        self._adjacencias.setdefault(origem, []).append(Rota(destino, peso, descricao))
        self._adjacencias.setdefault(destino, [])

    def menor_caminho(self, origem: str, destino: str) -> Tuple[float, List[str]]:
        if origem not in self._adjacencias or destino not in self._adjacencias:
            raise KeyError('Origem e destino devem pertencer ao grafo.')

        distancias = {vertice: inf for vertice in self._adjacencias}
        predecessores: Dict[str, str] = {}
        distancias[origem] = 0.0
        fila: List[Tuple[float, str]] = [(0.0, origem)]

        while fila:
            distancia_atual, atual = heappop(fila)
            if distancia_atual != distancias[atual]:
                continue
            if atual == destino:
                break
            for rota in self._adjacencias[atual]:
                candidata = distancia_atual + rota.peso
                if candidata < distancias[rota.destino]:
                    distancias[rota.destino] = candidata
                    predecessores[rota.destino] = atual
                    heappush(fila, (candidata, rota.destino))

        if distancias[destino] == inf:
            raise ValueError(f'Nao existe rota de {origem} para {destino}.')

        caminho = [destino]
        while caminho[-1] != origem:
            caminho.append(predecessores[caminho[-1]])
        caminho.reverse()
        return distancias[destino], caminho


def adicionar_rotas(grafo: GrafoLogistico, rotas: List[Tuple[str, str, float, str]]) -> None:
    for origem, destino, peso, descricao in rotas:
        grafo.adicionar_rota(origem, destino, peso, descricao)


# GM: Fluxo de materiais e utilidades da Estacao de Hidrogenio (DAG)
fluxo_materiais = GrafoLogistico()
rotas_materiais = [
    ('Portaria', 'Balança e Controle de Entrada', 25.0, 'liberacao e pesagem de carretas de H2'),
    ('Balança e Controle de Entrada', 'Pátio de Descarga de Tube Trailers', 45.0, 'estacionamento e aterramento de seguranca'),
    ('Pátio de Descarga de Tube Trailers', 'SUP-100 (Inlet Header)', 18.0, 'acoplamento de mangotes ao manifold de suprimento'),
    ('Pátio de Descarga de Tube Trailers', 'Central de Utilidades (N2 / Glicol)', 40.0, 'recebimento de N2 de purga e fluido refrigerante'),
    ('SUP-100 (Inlet Header)', 'TK-101 (Banco LP - 350 bar)', 10.0, 'alimentacao do banco LP via XV-101'),
    ('SUP-100 (Inlet Header)', 'TK-102 (Banco MP - 700 bar)', 12.0, 'alimentacao do banco MP via XV-102'),
    ('SUP-100 (Inlet Header)', 'TK-103 (Banco HP - 1000 bar)', 15.0, 'alimentacao do banco HP via XV-103'),
    ('TK-101 (Banco LP - 350 bar)', 'XV-104 (Válvula Direcional)', 8.0, 'descarga da linha LP de 350 bar'),
    ('TK-102 (Banco MP - 700 bar)', 'XV-104 (Válvula Direcional)', 6.0, 'descarga da linha MP de 700 bar'),
    ('TK-103 (Banco HP - 1000 bar)', 'XV-104 (Válvula Direcional)', 8.0, 'descarga da linha HP de 1000 bar'),
    ('Central de Utilidades (N2 / Glicol)', 'HEX-201 (Chiller -40°C)', 35.0, 'circulacao de fluido termico refrigerante'),
    ('XV-104 (Válvula Direcional)', 'HEX-201 (Chiller -40°C)', 14.0, 'resfriamento criogenico do H2 via XV-201'),
    ('HEX-201 (Chiller -40°C)', 'DISP-300 (Dispensador)', 16.0, 'transferencia de H2 pre-resfriado via XV-301'),
    ('DISP-300 (Dispensador)', 'Estoque Veicular FCEV 700 bar', 6.0, 'abastecimento SAE J2601 via BV-301'),
    ('DISP-300 (Dispensador)', 'Estoque Veicular FCEV 350 bar', 8.0, 'abastecimento de onibus e veiculos pesados'),
    ('Estoque Veicular FCEV 700 bar', 'Pátio de Expedição de Veículos', 12.0, 'liberacao de veiculos leves abastecidos'),
    ('Estoque Veicular FCEV 350 bar', 'Pátio de Expedição de Veículos', 15.0, 'liberacao de veiculos pesados abastecidos'),
]
adicionar_rotas(fluxo_materiais, rotas_materiais)

distancia, caminho = fluxo_materiais.menor_caminho('Portaria', 'Pátio de Expedição de Veículos')
print('Fluxo de materiais até a expedição')
print('  ' + ' -> '.join(caminho))
print(f'  Distância didática total: {distancia:.0f} m')
assert 'XV-104 (Válvula Direcional)' in caminho
assert 'HEX-201 (Chiller -40°C)' in caminho
assert 'DISP-300 (Dispensador)' in caminho

print('\nRotas dos insumos até a expedição')
for insumo in [
    'TK-101 (Banco LP - 350 bar)',
    'TK-102 (Banco MP - 700 bar)',
    'TK-103 (Banco HP - 1000 bar)',
    'Central de Utilidades (N2 / Glicol)',
]:
    dist_ins, cam_ins = fluxo_materiais.menor_caminho(insumo, 'Pátio de Expedição de Veículos')
    rota_txt = ' -> '.join(cam_ins)
    print(f'  {insumo}: {dist_ins:.0f} m ({rota_txt})')


# GV: Circulacao interna de veiculos (Anel Viario de Sentido Unico)
circulacao_veiculos = GrafoLogistico()
anel_viario = [
    ('Portaria', 'Balança de entrada', 25.0, 'entrada e pesagem de veiculos'),
    ('Balança de entrada', 'Pátio de descarregamento de Tube Trailers', 45.0, 'estacionamento e descarga de carretas de H2'),
    ('Pátio de descarregamento de Tube Trailers', 'Área de compressores e armazenamento', 40.0, 'rota tecnica de servico'),
    ('Área de compressores e armazenamento', 'Ilha de dispensação (Dispensers)', 45.0, 'circulacao interna de veiculos'),
    ('Ilha de dispensação (Dispensers)', 'Pátio de expedição de veículos', 30.0, 'rota pos-abastecimento de veiculos FCEV'),
    ('Pátio de expedição de veículos', 'Balança de saída', 25.0, 'pesagem final e liberacao de saida'),
    ('Balança de saída', 'Portaria', 30.0, 'saida de veiculos da planta'),
]
adicionar_rotas(circulacao_veiculos, anel_viario)

distancia_saida, rota_saida = circulacao_veiculos.menor_caminho('Ilha de dispensação (Dispensers)', 'Portaria')
print('\nCirculação de veículo após o carregamento')
print('  ' + ' -> '.join(rota_saida))
print(f'  Distância didática total: {distancia_saida:.0f} m')
assert rota_saida == ['Ilha de dispensação (Dispensers)', 'Pátio de expedição de veículos', 'Balança de saída', 'Portaria']


Fluxo de materiais até a expedição
  Portaria -> Balança e Controle de Entrada -> Pátio de Descarga de Tube Trailers -> SUP-100 (Inlet Header) -> TK-101 (Banco LP - 350 bar) -> XV-104 (Válvula Direcional) -> HEX-201 (Chiller -40°C) -> DISP-300 (Dispensador) -> Estoque Veicular FCEV 700 bar -> Pátio de Expedição de Veículos
  Distância didática total: 154 m

Rotas dos insumos até a expedição
  TK-101 (Banco LP - 350 bar): 56 m (TK-101 (Banco LP - 350 bar) -> XV-104 (Válvula Direcional) -> HEX-201 (Chiller -40°C) -> DISP-300 (Dispensador) -> Estoque Veicular FCEV 700 bar -> Pátio de Expedição de Veículos)
  TK-102 (Banco MP - 700 bar): 54 m (TK-102 (Banco MP - 700 bar) -> XV-104 (Válvula Direcional) -> HEX-201 (Chiller -40°C) -> DISP-300 (Dispensador) -> Estoque Veicular FCEV 700 bar -> Pátio de Expedição de Veículos)
  TK-103 (Banco HP - 1000 bar): 56 m (TK-103 (Banco HP - 1000 bar) -> XV-104 (Válvula Direcional) -> HEX-201 (Chiller -40°C) -> DISP-300 (Dispensador) -> Estoque Veicular F